# Daily Fact Generator – Car Workshop

Generates **one day** of fact data as parquet on the volume, for Auto Loader to pick up.

The generation logic lives in **`simulator/fact_generation.py`** (shared with the
local Docker lab — see `simulator/README.md`); this notebook only resolves widgets,
loads FK pools from `car_workshop.dim.*` and calls `generate_day`.

Widgets: `TARGET_DATE` (blank = yesterday), `SCALE_FACTOR` (blank = 1.0 ≈ 100K rows/day).
Determinism: seed = TARGET_DATE, each day owns a disjoint 10M ID block per table —
re-running a date reproduces identical rows (but writes duplicate files).

In [ ]:
from datetime import date, timedelta

from simulator.fact_generation import DAILY_BASE_ROWS, dims_from_frames, generate_day

try:
    dbutils.widgets.text('TARGET_DATE', '', 'Target date (YYYY-MM-DD, blank = yesterday)')
    dbutils.widgets.text('SCALE_FACTOR', '', 'Scale factor (blank = 1.0)')
    _raw_date = dbutils.widgets.get('TARGET_DATE').strip()
    _raw_scale = dbutils.widgets.get('SCALE_FACTOR').strip()
except NameError:  # running outside Databricks
    _raw_date, _raw_scale = '', ''
TARGET_DATE = date.fromisoformat(_raw_date) if _raw_date else date.today() - timedelta(days=1)
SCALE_FACTOR = float(_raw_scale) if _raw_scale else 1.0  # 1.0 = ~100K rows/day

CATALOG = 'car_workshop'
FACT_OUTPUT_DIR = f'/Volumes/{CATALOG}/fact/fact_files'

_est = sum(max(int(n * SCALE_FACTOR), 1) for n in DAILY_BASE_ROWS.values())
print(f'TARGET_DATE  = {TARGET_DATE}')
print(f'SCALE_FACTOR = {SCALE_FACTOR}  (~{_est:,} rows + schedules)')
print(f'OUTPUT       = {FACT_OUTPUT_DIR}')

## Load dimensions (generated once by `initial_dims.ipynb`)

In [ ]:
def dim_column(table, col):
    return spark.table(f'{CATALOG}.dim.{table}').select(col).toPandas()[col].to_numpy()


# raises RuntimeError if any dim is empty - run initial_dims + dim ingestion first
dims = dims_from_frames(
    locations=spark.table(f'{CATALOG}.dim.dim_locations').select('location_id', 'type').toPandas(),
    employees=(spark.table(f'{CATALOG}.dim.dim_employees')
               .select('employee_id', 'position', 'location_id').toPandas()),
    customer_ids=dim_column('dim_customers', 'customer_id'),
    vehicle_ids=dim_column('dim_vehicles', 'vehicle_id'),
    product_ids=dim_column('dim_products', 'product_id'),
    service_ids=dim_column('dim_services', 'service_id'),
    supplier_ids=dim_column('dim_suppliers', 'supplier_id'),
)
print(f"dims loaded: {len(dims['customer_ids']):,} customers, "
      f"{len(dims['vehicle_ids']):,} vehicles, {len(dims['product_ids']):,} products, "
      f"{len(dims['employee_ids']):,} employees")

## Generate all 13 fact tables

In [ ]:
import time

import pandas as pd

_t0 = time.time()
stats = generate_day(TARGET_DATE, SCALE_FACTOR, FACT_OUTPUT_DIR, dims)

summary = pd.DataFrame(stats)
print(f'TARGET_DATE = {TARGET_DATE} | SCALE_FACTOR = {SCALE_FACTOR}')
print(f'TOTAL: {summary["rows"].sum():,} rows in {time.time() - _t0:.1f}s -> {FACT_OUTPUT_DIR}')
print('Next: run autoloader.ipynb to ingest into car_workshop.fact.*')